In [ ]:
import json
import pandas as pd
from groq import Groq  # Assuming you are using Groq based on your previous code
from dotenv import load_dotenv
load_dotenv()
import os
api_key=os.getenv("api_key")
# Initialize your client (make sure your api_key is loaded)
client = Groq(api_key=api_key)

# 1. Load your result data into a DataFrame
with open(r"..\testing\result.json", "r", encoding="utf-8") as f:
    data = json.load(f)

df = pd.DataFrame(data)

GROUNDEDNESS_PROMPT_TEMPLATE = """
You are an objective evaluation judge. Your task is to evaluate the GROUNDEDNESS of an AI-generated response based *only* on the provided context.

Evaluation Criteria:
- GROUNDEDNESS: Determine whether the claims made in the response are entirely supported by the provided context. 
- Do not give credit for information that relies on external knowledge or assumptions outside the context.
- Penalize unsupported claims, hallucinations, or information that contradicts the context.
- If the context does not contain enough information and the assistant correctly states it cannot answer, give it a high groundedness score.

Provide your evaluation in this exact format:
Score: <1 to 5 integer>
Reason: <brief explanation>

---
Context:
{context}

Assistant Response:
{response}
"""
def evaluate_groundedness(context, response):
    prompt = GROUNDEDNESS_PROMPT_TEMPLATE.format(context=context, response=response)
    try:
        completion = client.chat.completions.create(
            model="llama-3.3-70b-versatile",  # Or your chosen judge model
            messages=[{"role": "user", "content": prompt}],
            temperature=0
        )
        return completion.choices[0].message.content.strip()
    except Exception as e:
        return f"Error: {e}"

# --- Run Evaluation Across the DataFrame ---

print("--- Running Groundedness & Relevance Judges ---")

groundedness_results = []


for index, row in df.iterrows():
    print(f"Evaluating row {index + 1} (ID: {row['id']})...")
    
    # Run Groundedness Judge
    g_result = evaluate_groundedness(row['context'], row['Assistant_Response'])
    groundedness_results.append(g_result)


# Assign results back into the DataFrame
df['Groundedness_Evaluation'] = groundedness_results


# Display the updated dataframe with evaluations
pd.set_option("display.max_colwidth", None)
display(df[['id', 'question', 'Assistant_Response', 'Groundedness_Evaluation']])
